In [8]:
# from gulps.synthesis_plugin import GulpsSynthesisPlugin
import numpy as np
from monodromy.render import _plot_coverage_set
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.circuit.library import (
    CXGate,
    RZXGate,
    UGate,
    UnitaryGate,
    XXPlusYYGate,
    iSwapGate,
)
from qiskit.circuit.random import random_circuit
from qiskit.quantum_info import Operator, average_gate_fidelity
from qiskit.quantum_info.random import random_unitary
from qiskit.transpiler import (
    InstructionProperties,
    PassManager,
    Target,
    generate_preset_pass_manager,
)
from qiskit.transpiler.passes import Optimize1qGatesDecomposition
from tqdm import tqdm
from weylchamber import c1c2c3

from gulps.gulps_synthesis import GulpsDecomposer
from gulps.synthesis_pass import GulpsDecompositionPass
from gulps.utils.invariants import GateInvariants
from gulps.utils.logging_config import logger

# logger.setLevel("DEBUG")
logger.setLevel("INFO")


In [9]:
from qiskit.synthesis.two_qubit.xx_decompose import XXDecomposer

N = 100
slope, offset = 1e-10, 1e-12

basis_fidelity = {
    strength: 1.0 - (slope * strength / (np.pi / 2) + offset)
    for strength in [
        # np.pi / 2,
        np.pi / 4,
        # np.pi / 6,
        np.pi / 8,
        np.pi / 12,
        # np.pi / 16,
        # np.pi / 24,
    ]
}
xx_decomposer = XXDecomposer(basis_fidelity)
fidelities = []
for idx in tqdm(range(N)):
    u = random_unitary(4, seed=idx)
    v = Operator(xx_decomposer(u))
    fid = average_gate_fidelity(u, v)
    fidelities.append(fid)

    if fid < 1 - 1e-6:
        print(f"Unitary {idx} fidelity is low: {fid:.8f}")
        print("Canonical invariants:")
        print("U:", c1c2c3(u))
        print("V:", c1c2c3(v))
        print("\n")


100%|██████████| 100/100 [00:01<00:00, 87.26it/s]


In [10]:
gate_set = [
    # CXGate(),
    CXGate().power(1 / 2),
    # CXGate().power(1 / 3),
    CXGate().power(1 / 4),
    CXGate().power(1 / 6),
    # CXGate().power(1 / 8),
    # CXGate().power(1 / 12),
]
costs = [1 - v for v in basis_fidelity.values()]
decomposer = GulpsDecomposer(gate_set, costs, precompute_polytopes=True)
# if hasattr(decomposer.isa, "coverage_set"):
#     _plot_coverage_set(decomposer.isa.coverage_set)

In [11]:
fidelities = []

for idx in tqdm(range(N)):
    u = random_unitary(4, seed=idx)
    v = Operator(decomposer(u))
    fid = average_gate_fidelity(u, v)
    fidelities.append(fid)

    if fid < 1 - 1e-6:
        print(f"Unitary {idx} fidelity is low: {fid:.8f}")
        print("Canonical invariants:")
        print("U:", c1c2c3(u))
        print("V:", c1c2c3(v))
        print("\n")
        continue

# Summary statistics
fidelities = np.array(fidelities)
print(f"\nSummary across {len(fidelities)} samples:")
print(f"  Median fidelity: {np.median(fidelities)}")
print(f"  Mean fidelity:   {np.mean(fidelities)}")
print(f"  Minimum fidelity:{np.min(fidelities)}")

  0%|          | 0/100 [00:00<?, ?it/s]

100%|██████████| 100/100 [00:05<00:00, 16.85it/s]


Summary across 100 samples:
  Median fidelity: 1.0000000000000007
  Mean fidelity:   0.9999999999785194
  Minimum fidelity:0.9999999982704034


In [12]:
# # %reload_ext  snakeviz
# %load_ext snakeviz
# import cProfile

# u = random_unitary(4, seed=0)
# cProfile.run("decomposer._run(u)", "profile_timings/temph.prof")